# 02 — Limpieza (Data Cleaning)

**Objetivo:** Eliminar filas basura, columnas inservibles, y preparar los datos para analisis.

Datos crudos del ERP siempre vienen sucios:
- Filas vacias que Excel arrastra
- Columnas que nunca se llenaron
- Fechas en formato texto
- Valores inconsistentes

**Input:** `data/inventory_anon.xlsx` — archivo anonimizado

**Output:** DataFrames limpios listos para enriquecimiento

---
## 2.1 - Cargar datos

In [1]:
import pandas as pd

In [2]:
PATH_FILE="../data/inventory_anon.xlsx"

In [3]:
sheets = pd.read_excel(PATH_FILE, sheet_name=["Inventory by Storeroom", "OPEN ORDER"])
inv = sheets["Inventory by Storeroom"]
orders = sheets["OPEN ORDER"]

---
## 2.2 — Eliminar filas basura (inventory)

3,798 filas (18.9%) tienen 50+ columnas nulas. Son basura del Excel.


In [4]:
nulls_per_row = inv.isnull().sum(axis=1)
nulls_per_row

0         3
1         3
2         3
3         3
4         3
         ..
20042    61
20043    61
20044    61
20045    61
20046    61
Length: 20047, dtype: int64

In [5]:
sparse = nulls_per_row[nulls_per_row > 50].index
sparse

RangeIndex(start=16249, stop=20047, step=1)

In [6]:
inv = inv.drop(sparse)
inv

,Region Name,Country Name,Operation Name,Plant Code,Plant Name,Currency,Exchange Rate,Blanket Number,Commodity Code,Commodity,...,Manufacturer Code,Manufacturer Name,Manufacturer Part,Total On Order,Last Modified,In ERP,Average Daily Usage,Trend,Unnamed: 60,Unnamed: 61
0,North America,Mexico,OP-1,PLCODE-1,PLANT-1,EUR,1.0,EURO,MACHINE SPARES,2.0,...,MFG-CODE-001,MFG-001,MFG-PART-0001,0.0,2019-02-19 01:09:17.087,N,0.000000,No Trend,NaN,EUR
1,North America,Mexico,OP-1,PLCODE-1,PLANT-1,EUR,1.0,EURO,PERISHABLE,3.0,...,MFG-CODE-002,MFG-002,MFG-PART-0002,0.0,2019-02-19 01:09:17.087,N,0.000000,No Trend,NaN,EUR
2,North America,Mexico,OP-1,PLCODE-1,PLANT-1,EUR,1.0,EURO,PERISHABLE,3.0,...,MFG-CODE-002,MFG-002,MFG-PART-0003,0.0,2019-02-19 01:09:17.087,N,0.000000,No Trend,NaN,EUR
3,North America,Mexico,OP-1,PLCODE-1,PLANT-1,EUR,1.0,EURO,PERISHABLE,3.0,...,MFG-CODE-002,MFG-002,MFG-PART-0004,0.0,2019-02-19 01:09:17.087,N,0.000000,No Trend,NaN,EUR
4,North America,Mexico,OP-1,PLCODE-1,PLANT-1,EUR,1.0,EURO,PERISHABLE,3.0,...,MFG-CODE-002,MFG-002,MFG-PART-0005,0.0,2019-02-19 01:09:17.087,N,0.000000,No Trend,NaN,EUR
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16244,North America,Mexico,OP-1,PLCODE-1,PLANT-1,USD,1.0,USD,PPE,8.0,...,MFG-CODE-173,MFG-170,MFG-PART-6262,0.0,2019-02-19 01:09:17.087,N,0.000000,No Trend,NaN,USD
16245,North America,Mexico,OP-1,PLCODE-1,PLANT-1,USD,1.0,USD,PPE,8.0,...,MFG-CODE-173,MFG-170,MFG-PART-5232,0.0,2019-02-19 01:09:17.087,N,0.000000,No Trend,NaN,USD
16246,North America,Mexico,OP-1,PLCODE-1,PLANT-1,USD,1.0,USD,PPE,8.0,...,MFG-CODE-173,MFG-170,MFG-PART-5232,0.0,2019-02-19 01:09:17.087,N,0.000000,No Trend,NaN,USD
16247,North America,Mexico,OP-1,PLCODE-1,PLANT-1,USD,1.0,USD,PPE,8.0,...,MFG-CODE-173,MFG-170,MFG-PART-5232,0.0,2019-02-19 01:09:17.087,N,0.000000,No Trend,NaN,USD


In [7]:
print("Shape despues de limpiar filas:", inv.shape)

Shape despues de limpiar filas: (16249, 62)


---
## 2.3 - Eliminar columnas basura (inventory)


- Columnas 100% nulas: `Critcal Spares`, `Unnamed: 60`
- Columna basura de Excel: `Unnamed: 61`



In [8]:
null_per_col_pct = inv.isnull().sum() / len(inv)
null_per_col_pct

Region Name            0.0
Country Name           0.0
Operation Name         0.0
Plant Code             0.0
Plant Name             0.0
                      ... 
In ERP                 0.0
Average Daily Usage    0.0
Trend                  0.0
Unnamed: 60            1.0
Unnamed: 61            0.0
Length: 62, dtype: float64

In [10]:
null_full = null_per_col_pct[null_per_col_pct == 1.0].index.to_list()
null_full

['Critcal Spares', 'Unnamed: 60']

In [11]:
null_full.append("Unnamed: 61")
null_full

['Critcal Spares', 'Unnamed: 60', 'Unnamed: 61']

In [12]:
print("Columnas a eliminar:", null_full)
inv = inv.drop(columns=null_full)
print("Shape despues de limpiar columnas:", inv.shape)

Columnas a eliminar: ['Critcal Spares', 'Unnamed: 60', 'Unnamed: 61']
Shape despues de limpiar columnas: (16249, 59)


---
## 2.4 - Eliminar columnas basura (orders)

Orders tiene 6 columnas 100% nulas:
`Avg. Mon. Usage`, `Analyst Code`, `Annual Usage`, `Last Used Date`, `Last Received Date`, `Critical`

In [14]:
null_col_pct_orders = orders.isnull().sum() / len(orders)
null_col_pct_orders

Plant Code            0.000000
Item Number           0.000000
Item Description      0.000000
UOM                   0.000000
PO Number             0.001049
Issue Date            0.000000
Order Qty             0.000000
Open Qty              0.000000
Required Date         0.000262
BOH                   0.000000
Avg. Mon. Usage       1.000000
Price                 0.000000
Supplier Code         0.000000
Supplier Name         0.000000
Analyst Code          1.000000
Annual Usage          1.000000
Last Used Date        1.000000
Last Received Date    1.000000
Critical              1.000000
Last Modified         0.000000
Vendor PO             0.054550
dtype: float64

In [15]:
full_nulll_orders = null_col_pct_orders[null_col_pct_orders == 1.0].index.to_list()
full_nulll_orders

['Avg. Mon. Usage',
 'Analyst Code',
 'Annual Usage',
 'Last Used Date',
 'Last Received Date',
 'Critical']

In [17]:
print("Columnas a eliminar:", full_nulll_orders)

orders = orders.drop(columns=full_nulll_orders)
print("Shape despues de limpiar columnas:", orders.shape)

Columnas a eliminar: ['Avg. Mon. Usage', 'Analyst Code', 'Annual Usage', 'Last Used Date', 'Last Received Date', 'Critical']
Shape despues de limpiar columnas: (3813, 15)


---
## 2.5 — Convertir fechas

Algunas columnas de fecha puede que pandas las haya leido bien (datetime64) o como texto (object). Si alguna es tipo object convertir.

```python
pd.to_datetime(df["columna"], errors="coerce")
```

- `errors="coerce"` — si un valor no se puede convertir, lo pone como NaT (Not a Time) en vez de dar error

In [34]:
print("=== Inventory: columnas datetime ===") 
print(inv.select_dtypes(include="datetime").columns.to_list())


=== Inventory: columnas datetime ===
['Item Creation Date', 'Last Used', 'Last Recieved', 'Last Modified']


In [35]:
print("\n=== Orders: columnas datetime ===")                                                      
print(orders.select_dtypes(include="datetime").columns.tolist()) 


=== Orders: columnas datetime ===
['Issue Date', 'Required Date ', 'Last Modified']


---
## 2.6 — Verificar limpieza

1. Cuantas filas y columnas tiene ahora `inv`?
2. Cuantas filas y columnas tiene ahora `orders`? 
3. Ya no hay columnas 100% nulas?
4. Ya no hay filas con 50+ nulos?


In [36]:
print("===inventoy===")
print("shape:", inv.shape)
print("columnas 1005 nulas:", inv.columns[inv.isnull().all()].to_list())
print("filas con 50+ nulos:", (inv.isnull().sum(axis=1) > 50).sum())

print("===orders===")
print("shape:", orders.shape)
print("columnas 100% nulas:", orders.columns[orders.isnull().all()].to_list())

===inventoy===
shape: (16249, 59)
columnas 1005 nulas: []
filas con 50+ nulos: 0
===orders===
shape: (3813, 15)
columnas 100% nulas: []


---
## 2.7  Conclusiones


Inventory:                                                                                      
- Filas eliminadas: 3,798 (de 20,047 a 16,249)
- Columnas eliminadas: 3 (de 62 a 59)                                                           
- Columnas eliminadas fueron: Critcal Spares, Unnamed: 60, Unnamed: 61
                                                                                                
Orders:                                                                                         
- Filas eliminadas: 0
- Columnas eliminadas: 6 (de 21 a 15)                                                           
- Columnas eliminadas fueron: Avg. Mon. Usage, Analyst Code, Annual Usage, Last Used Date, Last 
  Received Date, Critical  
